# Peer-to-Peer (Custom Swarm) Multi-Agent Memory Evaluation

## The Problem

In a sequential peer-to-peer multi-agent system there is no central coordinator — agents run sequentially and share context through common working memory. Context propagation failures are harder to detect: if the first peer writes stale or incomplete analysis, every downstream peer inherits that error. When the brief changes after the first run, there's no hub to re-dispatch — each peer must independently reconcile new instructions with what earlier peers already committed to memory.

## What This Notebook Does

We run a **restaurant industry market research** peer-to-peer swarm through two sessions:

1. **Simple Session** — One run with a stable research brief. All three peers build on each other's work sequentially.
2. **Session with Feedback** — First run with the baseline brief. Then the user expands the scope (US → all of North America) and the swarm runs again with shared memory still holding the first run's analysis. Tests whether peers reconcile or inherit stale context.

The metrics tell us what happened — no manual labeling needed.

## Architecture

```
              ┌──────────────────────────┐
              │   Research Brief (User)   │
              └────────────┬─────────────┘
                           │
                           ▼
              ┌──────────────────────────┐
              │  market_trends_analyst    │  Peer 1 — reads memory,
              │                          │  analyses trends, appends
              └────────────┬─────────────┘
                           │
                           ▼
              ┌──────────────────────────┐
              │ customer_insights_analyst │  Peer 2 — reads memory
              │                          │  (sees peer 1's output),
              └────────────┬─────────────┘  adds insights, appends
                           │
                           ▼
              ┌──────────────────────────┐
              │   strategy_synthesizer   │  Peer 3 — reads memory
              │                          │  (sees both prior), synthesises
              └──────────────────────────┘
                           │
              ┌────────────▼─────────────┐
              │  Shared Memory (list)    │
              │  Persists across runs    │
              └──────────────────────────┘
```

**Flow:**
- No coordinator — peers run in a fixed sequence
- Each peer reads shared memory, runs the LLM, appends its response
- Custom swarm = plain Strands `Agent` objects + a shared Python list (no `Swarm` class)
- Memory persists across sessions to test scope-change scenarios

## Metrics Evaluated

**Memory Context Metrics** (LLM-as-judge, 1-5 scale):

| Metric | What it measures |
|--------|------------------|
| Context Freshness | Is the peer working with the latest information? |
| Handoff Completeness | Did the handoff include all facts the peer needs? |
| Context Utilization | Did the peer use the context it read from memory? |
| State Consistency | Do all peers agree on key facts (market size, target, competitors)? |
| Memory Write Accuracy | Is what the peer wrote to memory factually correct? |
| Redundant Context | How much repeated/irrelevant context was transferred? |
| C2 Alignment | Cosine similarity of peer outputs (Titan embeddings) |

**Memory Latency Metrics** (timers + token counts):

| Metric | What it measures |
|--------|------------------|
| Memory Read/Write Latency | Time for memory operations |
| Coordination Latency % | Fraction of time spent on coordination vs reasoning |
| Coordination Token % | Fraction of tokens spent on context vs generation |

## Step 1 — Setup

In [ ]:
!pip install -qr requirements.txt

In [ ]:
import os
import time
import logging
from IPython.display import display, Markdown

from strands import Agent

from metrics_collector import MetricsCollector
from eval_helpers import (
    format_memory, print_memory,
    compute_embeddings, c2_alignment_report,
)
from model_config import (
    AGENT_MODEL_ID,
    MARKET_TRENDS_PROMPT, CUSTOMER_INSIGHTS_PROMPT, STRATEGY_SYNTH_PROMPT,
)

region = os.getenv("AWS_REGION", "us-west-2")

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s",
                    datefmt="%Y-%m-%d %H:%M:%S")
logger = logging.getLogger("peer-to-peer-metrics")

## Step 2 — Shared Memory (Python list)

Same pattern as the hub-and-spoke notebook. Shared memory is a simple list that peers read from and append to. Each entry is tagged with the agent name.

`format_memory()` lives in `eval_helpers.py` and is used by both notebooks.

In [ ]:
# format_memory() is imported from eval_helpers
# It takes a list of {agent, role, content} entries and returns a readable string

print(format_memory([
    {"agent": "example", "role": "assistant", "content": "This is how memory looks when formatted."},
]))

## Step 3 — Peer Runner

Unlike the hub-spoke notebook which uses a `ListMemoryHook` attached to each spoke, the peer-to-peer runner is a simple loop. For each peer we explicitly:
1. Record the handoff (the task is passed directly — no coordinator compression)
2. Read shared memory and record it as retrieved context
3. Run the agent with memory baked into the system prompt
4. Record the response, latency, and tokens
5. Append the response to shared memory

No hook abstraction — every step is explicit and visible.

In [ ]:
PEER_CONFIGS = [
    ("market_trends",      MARKET_TRENDS_PROMPT),
    ("customer_insights",  CUSTOMER_INSIGHTS_PROMPT),
    ("strategy_synth",     STRATEGY_SYNTH_PROMPT),
]

def run_peers(task: str, shared_memory: list, collector: MetricsCollector,
              turn_number: int, session_label: str) -> dict:
    """Run all peers sequentially. Returns {agent_name: response}."""
    collector.begin_turn(turn_number, task)
    responses = {}

    print(f"\n{'='*60}")
    print(f"[{session_label}] Turn {turn_number}: {task[:80]}...")
    print(f"{'='*60}")

    for name, prompt in PEER_CONFIGS:
        # 1. Record handoff
        collector.record_handoff(name, task)

        # 2. Read memory
        t0 = time.perf_counter()
        context = format_memory(shared_memory)
        collector.record_memory_read_latency(name, time.perf_counter() - t0)
        collector.record_retrieved_context(name, context)

        if context:
            logger.info(f"[{name}] Read {len(shared_memory)} entries from memory")

        # 3. Run the agent
        full_prompt = prompt
        if context:
            full_prompt += (
                f"\n\nShared memory from other agents:\n{context}"
                "\n\nUse this context. Reference specific details from other agents."
            )
        agent = Agent(name=name, model=AGENT_MODEL_ID, system_prompt=full_prompt)

        t0 = time.perf_counter()
        resp = agent(task)
        latency = time.perf_counter() - t0
        response_text = str(resp)

        # 4. Record response + latency + tokens
        collector.record_response(name, response_text)
        collector.record_agent_latency(name, latency)
        usage = getattr(resp, "usage", None) or {}
        collector.record_token_usage(name,
            usage.get("inputTokens", 0), usage.get("outputTokens", 0))

        # 5. Write to shared memory
        t0 = time.perf_counter()
        shared_memory.append({
            "agent": name,
            "role": "assistant",
            "content": response_text,
            "ts": time.time(),
        })
        collector.record_memory_write_latency(name, time.perf_counter() - t0)

        responses[name] = response_text
        print(f"\n{name}: {response_text[:200]}...")

    collector.end_turn()
    return responses

## Step 4 — Session Runner

Wraps `run_peers()` so each session gets a fresh shared memory list and collector. Returns both so we can inspect memory afterwards.

In [ ]:
def run_session(conversation: list, session_label: str) -> tuple:
    """Run a full session (one or more turns). Returns (collector, shared_memory)."""
    shared_memory = []  # fresh memory per session
    collector = MetricsCollector(region=region)

    for i, task in enumerate(conversation, start=1):
        run_peers(task, shared_memory, collector,
                  turn_number=i, session_label=session_label)

    return collector, shared_memory

## Step 5 — Simple Session

One run with a stable research brief. All peers build on each other. This is the baseline.

In [ ]:
simple_conversation = [
    "Analyse the fast-casual restaurant market in the United States. "
    "Key players include Chipotle, Panera Bread, and Sweetgreen. "
    "Target segment: health-conscious urban diners aged 25-45. "
    "Market size: $60 billion, growing at 8% annually.",
]

simple_metrics, simple_memory = run_session(simple_conversation, "simple")

### Simple Session — Context Flow Trace

Shows what each peer received (handoff + memory read) and produced (response). Follow the flow to see how context moves through the swarm.

In [ ]:
display(Markdown(simple_metrics.trace_report()))

### Simple Session — Shared Memory Contents

Raw contents of the shared memory list after the session completes.

In [ ]:
print_memory(simple_memory)

## Step 6 — Session with Feedback

First run with the baseline brief. Then the user expands scope (US → North America) and the swarm runs again. Shared memory from run 1 persists into run 2.

This tests:
- Do peers in run 2 pick up the expanded scope?
- Do they reconcile V1 analysis with the new V2 instructions, or inherit stale context?
- Does the final synthesiser produce a plan reflecting the updated scope?

In [ ]:
feedback_conversation = [
    "Analyse the fast-casual restaurant market in the United States. "
    "Key players include Chipotle, Panera Bread, and Sweetgreen. "
    "Target segment: health-conscious urban diners aged 25-45. "
    "Market size: $60 billion, growing at 8% annually.",

    "Actually, expand the analysis to cover all of North America, not just the United States. "
    "Include Canadian and Mexican fast-casual chains as well. "
    "The North American market size is $85 billion. "
    "Keep the same focus on health-conscious urban diners aged 25-45.",
]

feedback_metrics, feedback_memory = run_session(feedback_conversation, "feedback")

### Feedback Session — Context Flow Trace

In [ ]:
display(Markdown(feedback_metrics.trace_report()))

### Feedback Session — Shared Memory Contents

In [ ]:
print_memory(feedback_memory)

## Step 7 — Run LLM-as-Judge Evaluations

Runs Claude Opus as judge on every peer call in both sessions. Evaluates context freshness, handoff completeness, utilization, write accuracy, redundancy, and cross-agent consistency.

In [ ]:
print("Evaluating simple session...")
simple_metrics.evaluate_all()

print("Evaluating feedback session...")
feedback_metrics.evaluate_all()

### C2 Alignment (Titan embeddings)

Peer-to-peer specific metric: cosine similarity between each peer's final response. Low similarity = peers diverged in their understanding of the shared reality.

In [ ]:
# Build {agent_name: final_response} from the memory of each session
simple_final = {e["agent"]: e["content"] for e in simple_memory}
feedback_final = {e["agent"]: e["content"] for e in feedback_memory if e.get("role") == "assistant"}

# For feedback session, keep only the LAST response per peer (from run 2)
feedback_last = {}
for e in feedback_memory:
    feedback_last[e["agent"]] = e["content"]

simple_embeddings = compute_embeddings(simple_final, region=region)
feedback_embeddings = compute_embeddings(feedback_last, region=region)

## Step 8 — Results: Simple Session

In [ ]:
display(Markdown("## Simple Session\n\n" + simple_metrics.context_metrics_report()))
display(Markdown(simple_metrics.latency_metrics_report()))
display(Markdown(c2_alignment_report(simple_embeddings, "Simple Session")))

## Step 9 — Results: Session with Feedback

In [ ]:
display(Markdown("## Session with Feedback\n\n" + feedback_metrics.context_metrics_report()))
display(Markdown(feedback_metrics.latency_metrics_report()))
display(Markdown(c2_alignment_report(feedback_embeddings, "Feedback Session")))

## Step 10 — Side-by-Side Comparison

In [ ]:
display(Markdown(MetricsCollector.comparison_report(
    simple_metrics, feedback_metrics,
    label_a="Simple Session", label_b="Session with Feedback")))

## Visual Analysis

In [ ]:
import matplotlib.pyplot as plt
from eval_helpers import (
    plot_context_metrics_radar, plot_latency_breakdown,
    plot_session_comparison, plot_c2_heatmap
)

In [ ]:
fig = plot_context_metrics_radar(simple_metrics, "Simple Session")
plt.show()

fig = plot_context_metrics_radar(feedback_metrics, "Feedback Session")
plt.show()

In [ ]:
fig = plot_latency_breakdown(simple_metrics, "Simple Session")
plt.show()

fig = plot_latency_breakdown(feedback_metrics, "Feedback Session")
plt.show()

In [ ]:
fig = plot_session_comparison(simple_metrics, feedback_metrics,
                              "Simple Session", "Feedback Session")
plt.show()

In [ ]:
if simple_embeddings:
    fig = plot_c2_heatmap(simple_embeddings, "Simple Session")
    plt.show()

if feedback_embeddings:
    fig = plot_c2_heatmap(feedback_embeddings, "Feedback Session")
    plt.show()